In [ ]:
!pip install langchain langchain-experimental langchain-community langchain-openai openai chromadb pypdf sentence_transformers gradio langchain-together

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
import getpass


from langchain_community.utilities import WikipediaAPIWrapper, SerpAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


# =====================================================
# Step 0: API Keys
# =====================================================
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
os.environ["SERPAPI_KEY"] = getpass.getpass("Enter SERPAPI API Key: ")
SERPAPI_KEY = os.environ["SERPAPI_KEY"]

In [ ]:
#---------------------------------------
# Initialise LLM
#---------------------------------------

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens=500,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
from langchain_core.messages import HumanMessage
from IPython.display import display, Markdown

request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.

Not a mathematical puzzle, but more of a thought-provoking question
that requires intelligent insight.

Include in your question that the answer must be short.

Answer only with the question, no explanation.
"""

response = llm.invoke(
    [HumanMessage(content=request)]
)

question = response.content

display(Markdown(question))

Multiple LLM Providers

In [ ]:
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown

competitors = []
answers = []


def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(f"### {model_name}\n\n{answer}"))

In [ ]:
def ask_model(model_name, question):

    llm = ChatOpenAI(
        model=model_name,
        temperature=0,
        max_tokens=300,
        openai_api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1"
    )

    response = llm.invoke(question)

    return response.content

In [ ]:
models = [

    # OpenAI
    "openai/gpt-4o-mini",

    # Anthropic
    "anthropic/claude-sonnet-4",

    # Google
    "google/gemini-2.5-flash",

    # Deepseek
    "deepseek/deepseek-chat-v3",

    # Grok
    "x-ai/grok-4.3",

    # Groq hosted models
    "groq/llama-3.3-70b-versatile",

    # Meta
    "meta-llama/llama-3.3-70b-instruct",

]

In [ ]:
for model_name in models:

    try:
        answer = ask_model(model_name, question)
        record(model_name, answer)

    except Exception as e:
        print(f"{model_name} failed")
        print(e)

In [ ]:
messages = question


model_name = "anthropic/claude-sonnet-4"

answer = ask_model(model_name, messages)
record(model_name, answer)


model_name = "google/gemini-2.5-flash"

answer = ask_model(model_name, messages)
record(model_name, answer)


model_name = "x-ai/grok-4.3"

answer = ask_model(model_name, messages)
record(model_name, answer)

In [ ]:
print(len(competitors))
print(competitors)
print(answers)

In [ ]:
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

In [ ]:
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]
judge_llm = ChatOpenAI(
    model="google/gemini-2.5-pro",
    temperature=0,
    max_tokens=500,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
response = judge_llm.invoke(judge_messages)

print(response.content)

In [ ]:
results_dict = json.loads(response.content)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")